In [ ]:
# ============================================================
# STEP 1 : Import Libraries
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_DIR = PROJECT_ROOT
OUTPUT_DIR = OUTPUT_ROOT / "cwc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# STEP 3 : Load Clean Datasets
# ============================================================

bio = pd.read_csv(bio_file)

chem = pd.read_csv(chem_file)

phys = pd.read_csv(phys_file)

print("Biological :", bio.shape)
print("Chemical   :", chem.shape)
print("Physical   :", phys.shape)

In [ ]:
# ============================================================
# STEP 4 : Extract Station Information
# ============================================================

station_cols = [
    "Station",
    "Latitude",
    "Longitude",
    "State",
    "District",
    "Tehsil",
    "Basin"
]

bio_station = bio[station_cols].copy()

chem_station = chem[station_cols].copy()

phys_station = phys[station_cols].copy()

In [ ]:
# ============================================================
# STEP 5 : Combine All Stations
# ============================================================

stations = pd.concat(
    [
        bio_station,
        chem_station,
        phys_station
    ],
    ignore_index=True
)

print(stations.shape)

In [ ]:
# ============================================================
# STEP 6 : Standardize Station Metadata
# ============================================================

stations["Station"] = (
    stations["Station"]
    .astype(str)
    .str.strip()
    .str.upper()
)

stations["District"] = (
    stations["District"]
    .astype(str)
    .str.strip()
)

stations["Tehsil"] = (
    stations["Tehsil"]
    .astype(str)
    .str.strip()
)

stations["Latitude"] = (
    stations["Latitude"]
    .astype(float)
    .round(6)
)

stations["Longitude"] = (
    stations["Longitude"]
    .astype(float)
    .round(6)
)

In [ ]:
# ============================================================
# STEP 7 : Create Coordinate Key
# ============================================================

stations["Coordinate_Key"] = (

    stations["Latitude"].astype(str)

    + "_"

    + stations["Longitude"].astype(str)

)

In [ ]:
# ============================================================
# STEP 8 : Unique Monitoring Locations
# ============================================================

stations = (

    stations

    .drop_duplicates(

        subset=[
            "Station",
            "Latitude",
            "Longitude"
        ]

    )

    .reset_index(drop=True)

)

print("Unique Monitoring Locations :", len(stations))

In [ ]:
# ============================================================
# STEP 9 : Sort Stations
# ============================================================

stations = (
    stations
    .sort_values(
        by=[
            "State",
            "District",
            "Station",
            "Latitude",
            "Longitude"
        ]
    )
    .reset_index(drop=True)
)

stations.head()

In [ ]:
# ============================================================
# STEP 10 : Generate Station_UID
# ============================================================

stations["Station_UID"] = [
    f"GNG{i:04d}"
    for i in range(1, len(stations) + 1)
]

stations = stations[
    [
        "Station_UID",
        "Station",
        "Latitude",
        "Longitude",
        "State",
        "District",
        "Tehsil",
        "Basin",
        "Coordinate_Key"
    ]
]

stations.head()

In [ ]:
# ============================================================
# STEP 11 : Validation
# ============================================================

print("Unique Station_UID :", stations["Station_UID"].nunique())

print(
    "Unique Coordinates :",
    stations["Coordinate_Key"].nunique()
)

print("Total Stations :", len(stations))

In [ ]:
# ============================================================
# STEP 13 : Find Same Coordinates with Different Station Names
# ============================================================

coord_dups = (
    stations[
        stations.duplicated(subset=["Coordinate_Key"], keep=False)
    ]
    .sort_values(["Coordinate_Key", "Station"])
)

print("Rows involved :", len(coord_dups))

display(coord_dups)

In [ ]:
for coord, grp in coord_dups.groupby("Coordinate_Key"):

    print("=" * 100)
    print("Coordinate :", coord)
    print()

    display(
        grp[
            [
                "Station_UID",
                "Station",
                "Latitude",
                "Longitude",
                "State",
                "District"
            ]
        ]
    )

In [ ]:
station_name_map = {
    "BASANTPUR_CWC": "BASANTPUR",
    "KACHHLABRIDGE": "KACHHLA BRIDGE",
    "KARNPRAYAG CONFLUENCE D/S": "KARNAPRAYAG CONFLUENCE D/S",
}

stations["Station"] = stations["Station"].replace(station_name_map)

In [ ]:
stations = (
    stations
    .drop_duplicates(
        subset=["Station", "Latitude", "Longitude"]
    )
    .reset_index(drop=True)
)

In [ ]:
print(stations["Station_UID"].nunique())
print(stations["Coordinate_Key"].nunique())
print(len(stations))

In [ ]:
# ============================================================
# STEP 13 : Regenerate Station_UID
# ============================================================

stations = stations.reset_index(drop=True)

stations["Station_UID"] = [
    f"GNG{i:04d}"
    for i in range(1, len(stations) + 1)
]

stations = stations[
    [
        "Station_UID",
        "Station",
        "Latitude",
        "Longitude",
        "State",
        "District",
        "Tehsil",
        "Basin",
        "Coordinate_Key"
    ]
]

print("Total Stations :", len(stations))
stations.head()

In [ ]:
# ============================================================
# STEP 14 : Save Station Master
# ============================================================

station_master_file = OUTPUT_DIR / "Ganga_Station_Master.csv"

stations.to_csv(
    station_master_file,
    index=False
)

print("Saved :", station_master_file)

In [ ]:
dup_station = stations.groupby("Station").size()
dup_station = dup_station[dup_station > 1]

print("Duplicate station names:", len(dup_station))
display(
    stations[stations["Station"].isin(dup_station.index)]
    .sort_values(["Station", "Latitude", "Longitude"])
)

In [ ]:
# ============================================================
# STEP 15 : Distance Audit for Duplicate Station Names
# ============================================================

from math import radians, sin, cos, sqrt, atan2

# ------------------------------------------------------------
# Haversine Distance (meters)
# ------------------------------------------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371000  # Earth radius (m)

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


# ------------------------------------------------------------
# Duplicate Station Audit
# ------------------------------------------------------------

audit = []

dup_names = (
    stations.groupby("Station")
    .filter(lambda x: len(x) > 1)
)

for station, grp in dup_names.groupby("Station"):

    grp = grp.sort_values(["Latitude", "Longitude"]).reset_index(drop=True)

    for i in range(len(grp) - 1):

        r1 = grp.iloc[i]
        r2 = grp.iloc[i + 1]

        dist = haversine(
            r1["Latitude"],
            r1["Longitude"],
            r2["Latitude"],
            r2["Longitude"]
        )

        if dist <= 20:
            action = "MERGE"
        elif dist <= 100:
            action = "REVIEW"
        else:
            action = "KEEP"

        audit.append({
            "Station": station,
            "Lat1": r1["Latitude"],
            "Lon1": r1["Longitude"],
            "Lat2": r2["Latitude"],
            "Lon2": r2["Longitude"],
            "Distance_m": round(dist, 2),
            "Recommendation": action
        })

audit = pd.DataFrame(audit)

audit = audit.sort_values("Distance_m")

display(audit)

print("\nRecommendation Summary\n")
print(audit["Recommendation"].value_counts())

In [ ]:
# ============================================================
# STEP 16 : Save Audit Report
# ============================================================

audit_file = OUTPUT_DIR / "Station_Duplicate_Audit.csv"

audit.to_csv(
    audit_file,
    index=False
)

print("Saved :", audit_file)

In [ ]:
# ============================================================
# STEP 17 : Create Merge Key (Station Master)
# ============================================================

stations["Merge_Key"] = (

    stations["Station"].str.strip().str.upper()

    + "|"

    + stations["Latitude"].round(6).astype(str)

    + "|"

    + stations["Longitude"].round(6).astype(str)

)

In [ ]:
# ============================================================
# STEP 18 : Function to Inject Station_UID
# ============================================================

station_name_map = {
    "BASANTPUR_CWC": "BASANTPUR",
    "KACHHLABRIDGE": "KACHHLA BRIDGE",
    "KARNPRAYAG CONFLUENCE D/S": "KARNAPRAYAG CONFLUENCE D/S",
}

def add_station_uid(df, station_master):

    temp = df.copy()

    # ------------------------------
    # Standardize Station Names
    # ------------------------------

    temp["Station"] = (
        temp["Station"]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace(station_name_map)
    )

    # ------------------------------
    # Create Merge Key
    # ------------------------------

    temp["Merge_Key"] = (

        temp["Station"]

        + "|"

        + temp["Latitude"]
        .astype(float)
        .round(6)
        .astype(str)

        + "|"

        + temp["Longitude"]
        .astype(float)
        .round(6)
        .astype(str)

    )

    # ------------------------------
    # Merge UID
    # ------------------------------

    temp = temp.merge(

        station_master[
            [
                "Merge_Key",
                "Station_UID"
            ]
        ],

        on="Merge_Key",

        how="left"

    )

    return temp

In [ ]:
bio_uid = add_station_uid(bio, stations)

chem_uid = add_station_uid(chem, stations)

phys_uid = add_station_uid(phys, stations)

In [ ]:
# ============================================================
# STEP 20 : Validation
# ============================================================

print("Biological :", bio_uid["Station_UID"].isna().sum())
print("Chemical   :", chem_uid["Station_UID"].isna().sum())
print("Physical   :", phys_uid["Station_UID"].isna().sum())

In [ ]:
# ============================================================
# STEP 21 : Save UID Datasets
# ============================================================

bio_uid.to_csv(
    OUTPUT_DIR / "CWC_Biological_Ganga_UID.csv",
    index=False
)

chem_uid.to_csv(
    OUTPUT_DIR / "CWC_Chemical_Ganga_UID.csv",
    index=False
)

phys_uid.to_csv(
    OUTPUT_DIR / "CWC_Physical_Ganga_UID.csv",
    index=False
)

print("All UID datasets saved successfully.")